# ローカル検証パイプライン on Google Colab — AI Agent Security

ハイブリッド運用の GPU 層を Colab で回す。**ランタイム → ハードウェアアクセラレータ → GPU** を選ぶこと。
- `gpt_oss`（gpt-oss-20b Q4, 約12GB）: 無料 **T4(16GB)** で可。
- `gemma_4`（gemma-4-26B Q4, 約16GB）: **L4(24GB) か A100** 推奨（Colab Pro / Pay-As-You-Go）。

手順: ①依存導入 → ②SDK 取得 → ③検証コード取得 → ④GGUF 取得 → ⑤実行。

## ① 依存（llama.cpp は CUDA ビルド）

In [ ]:
!pip -q install gymnasium 'pydantic>=2' huggingface_hub kaggle
# CUDA 版 llama-cpp-python（数分。事前ビルド wheel があればそれが入る）
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
!pip -q install llama-cpp-python
import llama_cpp; print('llama_cpp OK')

## ② SDK 取得（downloads/aicomp_sdk_pkg/ に展開）
Kaggle API（`kaggle.json` を `/root/.kaggle/` に置く）か、コンペ zip をアップロードする。

In [ ]:
import os, zipfile, glob
os.makedirs('downloads/aicomp_sdk_pkg', exist_ok=True)
# 方法A: Kaggle API（要 kaggle.json）
# from google.colab import files; files.upload()  # kaggle.json を選択
# !mkdir -p /root/.kaggle && cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
# !kaggle competitions download -c ai-agent-security-multi-step-tool-attacks -p downloads/
# 方法B: 手元の downloads/ai-agent-security-multi-step-tool-attacks.zip をアップロード
# from google.colab import files; up = files.upload()
z = glob.glob('downloads/*.zip') + glob.glob('*.zip')
assert z, 'コンペ zip を downloads/ に置くか Kaggle API で取得してください'
with zipfile.ZipFile(z[0]) as f: f.extractall('downloads/aicomp_sdk_pkg')
print('extracted ->', sorted(os.listdir('downloads/aicomp_sdk_pkg'))[:5])

## ③ 検証コード取得
リポジトリを clone するか、`validation/` をアップロードする。

In [ ]:
# 例: git clone <YOUR_REPO_URL> .  （downloads/ は gitignore なので ② で別途取得済み）
# もしくは validation/ フォルダをアップロード。validation/__init__.py が見えれば OK。
import os; assert os.path.isdir('validation'), 'validation/ を配置してください'
print('validation OK')

## ④ GGUF 取得（初回のみ）

In [ ]:
MODEL = 'gpt_oss'  # or 'gemma_4'（L4/A100 推奨）。gemma は Google ライセンス同意が必要な場合あり
!python -m validation.download_models {MODEL}

## ⑤ 実行（公開 LB 相関 = public、汎化代理 = provenance）

In [ ]:
ATTACK = 'baseline'  # or 'path/to/your/attack.py'
!python -m validation.run_validation \
    --attack {ATTACK} --agent {MODEL} \
    --guardrails public,provenance \
    --candidates 30 --budget-s 600 --env gym \
    --candidates-out runs/cand_{MODEL}.json --report-out runs/report_{MODEL}.txt